# Surmounting Withdrawal to Initiate Fast Treatment With Naltrexone (SWIFT)

Prepared by:
: J M Maxwell, Center for Translational Data Science (CTDS), University of Chicago

:::{note} Attribution
:class: dropdown
This notebook uses data collected by [](doi:10.1001/jamanetworkopen.2024.9744) as part of their study entitled *Surmounting Withdrawal to Initiate Fast Treatment with Naltrexone (SWIFT): Improving the Real-World Effectiveness of Injection Naltrexone for Opioid Use Disorder*. The data have been archived by the authors at [NIDA Data Share](https://datashare.nida.nih.gov/study/nida-ctn-0097) and are accessible via the [HEAL Data Platform](10.60490/HDP01319).

The purpose of this notebook is to demonstrate how the data may be accessed and used for analysis, and is intended to be used as a jumping off point for researchers who may wish to use these data for their own secondary analyses. While some of the analyses below may recreate analyses presented in the original publication, they are not intended to replicate the original results and differences in analytic methods and software, selection/filtering of observations, and handling of missing data may yield results that differ from the original.

The work here was conducted without direct involvement of the original authors and therefore does not necessarily reflect the views or opinions of the authors, of the NIH HEAL Initiative&reg;, or of the Center for Translational Data Science (CTDS) at the University of Chicago.
:::

## About The Study

The study, *Surmounting Withdrawal to Initiate Fast Treatment With Naltrexone (SWIFT)*, investigates two methods of treatment with extended-release naltrexone (XR-NTX). The primary comparison between the two methods is if the Rapid Method treatment (5-7 days long) is non-inferior to the Standard Method (13 days long) in succesful initiation of XR-NTX, with a secondary comparisons on time from admission to dose to dropout, craving, withdrawal severity, retention, abstinence, and safety measures as measured from admission through the first two months of post-induction maintenance of XR-NTX. [(NCT: 04762537)](https://clinicaltrials.gov/study/NCT04762537) 

The Standard Procedure was defined as an approximately 5 day buprenorphine taper, a 7-10 day opioid free period, finished with a XR-naltrexone package insert. The Rapid Procedure was defined as 1 day of buprenorphine at the minimal necessary dosage, 1 day opioid free, finished with ascending low dosages of oral naltrexone and additional medications for opioid withdrawal.

## Setup

This notebook uses Python (tested with version 3.13) and relies on several modules and third-party packages. Thus we start by installing and/or importing these if necessary (click on the bar below to expand the cell and see the code).

In [ ]:
!pip install pandas openpyxl gen3 tableone scipy matplotlib -q
import os
import numpy as np
import pandas as pd
from pandas.api.types import CategoricalDtype
import zipfile
from tableone import TableOne
from scipy import stats
import matplotlib.pyplot as plt
from IPython.display import Markdown, Image, display

## Retrieving the Data

The data used here are archived at [NIDA Data Share](https://datashare.nida.nih.gov/study/nida-ctn-0097). Accessing data from [NIDA Data Share](https://datashare.nida.nih.gov/study/nida-ctn-0097) currently requires that you login to the Platform using the InCommon login option. After doing so, you may download the data directly from the [Study Page](10.60490/HDP01319) (under the Data tab) or you may download them using the `gen3` command, as we will do here.

Each file indexed by the Platform is assigned a globally unique identifier (GUID) that may be used to retrieve the file. In this case, the GUID is `8a0b2f16-4bd1-4033-acfe-b353917aeb20`.

:::{note}
We use an if condition to avoid downloading the file each time we re-run the notebook.
:::

In [ ]:
if not os.path.exists('sas-crf-data-files_nida-ctn-0097 v1-1.zip'):
    !gen3 drs-pull object dg.H34L/8a0b2f16-4bd1-4033-acfe-b353917aeb20

## Reading the Data

The downloaded file is a zip archive, so we need to unzip it before we can read the data. We then read two files into Pandas dataframes: one file containing the core variables and one containing the inpatient COWS scores.

In [ ]:
with zipfile.ZipFile('sas-crf-data-files_nida-ctn-0097 v1-1.zip', 'r') as archive:
    archive.extractall('ctn0097')

core = pd.read_sas('ctn0097/corevars.sas7bdat', encoding='utf-8')
ip = pd.read_sas('ctn0097/inpatient.sas7bdat', encoding='utf-8')

Here we do a little reformatting.

In [ ]:
core = (core
        .assign(AGE=core.AGE.astype('Int64'),
                SEX=core.SEX.astype('object').replace({1.0: 'Male',
                                                       2.0: 'Female'}).astype('category'),
                ETHNIC_CAT=core.ETHNIC_CAT.astype('object').replace({0.0: 'Non-Hispanic',
                                                                     1.0: 'Hispanic'}).astype('category'))
       )

## Secondary Outcome Analysis

While the primary outcome measured was the receipt of the first dose of XR-naltrexone during the inpatient period, the secondary outcome explore here is the Clinical Opiate Withdrawal Scalse (COWS) Scores over the first 15 days of the trial. 

In the table below we see that 415 individuals participated in the study and the corresponding demographic summaries of the individuals.

141 of the 225 (62.7%) assigned to the Rapid Procedure and 68 of the 190 (35.8%) assigned to the Standard Procedure ultimately received XR-naltrexone. Through non-inferiority tests and subgroup analyses it was determined that the rate of initiation success among the Rapid Procedure group was noninferior to the rate of initiation in the Standard Procedure group.


In [ ]:
columns = ['AGE', 'SEX', 'ETHNIC_CAT']
categorical = ['SEX', 'ETHNIC_CAT']
continuous = ['AGE']
groupby = 'TRT'
TableOne(core, columns=columns, categorical=categorical, continuous=continuous, groupby=groupby, pval=False)

The COWS score was measured as a binary for the presence of at least 1 moderate or above COWS score. In the figure and table below we can see that the mean and standard deviations of the Rapid and Standard Procedure were very similar throughout the course of the inpatient days, and that withdrawal did not differ significantly between procedure conditions. 

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(12, 6))

df2_viz = pd.DataFrame(ip.groupby(['INP_DAY', 'TRT'])['COW_SCORE_MAX'].agg(['mean', 'std'])).reset_index()
df2_viz = df2_viz[(df2_viz.INP_DAY > 0) & (df2_viz.INP_DAY <= 15)]

plt.errorbar(df2_viz[df2_viz.TRT == 'Standard'].INP_DAY - 0.1, df2_viz[df2_viz.TRT == 'Standard']['mean'],  df2_viz[df2_viz.TRT == 'Standard']['std'], marker='.', elinewidth=0.9)
plt.errorbar(df2_viz[df2_viz.TRT == 'Rapid'].INP_DAY + 0.1, df2_viz[df2_viz.TRT == 'Rapid']['mean'], df2_viz[df2_viz.TRT == 'Rapid']['std'], marker='.', elinewidth=0.9)

plt.xticks(np.linspace(1, 15, 15))
plt.yticks(np.linspace(0, 15, 6))
plt.ylabel('Daily maximum COWS Score, Mean')
plt.xlabel('Inpatient Day')

plt.title('Mean (SD) Daily Maximum COWS Score During the First 15 Days Treatment Procedure.')

plt.legend(["Standard Procedure", "Rapid Procedure"])

plt.show()

In [ ]:
df2_viz.round(2)

## Conclusions

As we have briefly shown in this notebook, the study by Bisaga et. al. and the paper by Shulman et. al. demonstrates how the Rapid Procedure for XR-naltrexone initiation was noninferior compared to the Standard Procedure in treatment intiation and patients' self reported withdrawal outcomes. Shulman et. al. conclude that the Rapid Procedure treatment initiation, while requiring more intensive medical engagement and monitoring, "could make XR-naltrexone a more viable treatment for patients with OUD."

## References

#### Paper
Shulman M, Greiner MG, Tafessu HM, et al. Rapid Initiation of Injection Naltrexone for Opioid Use Disorder: A Stepped-Wedge Cluster Randomized Clinical Trial. JAMA Netw Open. 2024;7(5):e249744. Published 2024 May 1. doi:10.1001/jamanetworkopen.2024.9744

#### Study

Bisaga A, Nunes E. Surmounting Withdrawal to Initiate Fast Treatment With Naltrexone (SWIFT). NIDA-CTN-0097. https://datashare.nida.nih.gov/study/nida-ctn-0097

#### HEAL Data Platform

Bisaga, Adam & Jr.,   (2025). Surmounting Withdrawal to Initiate Fast Treatment with Naltrexone (SWIFT): Improving the Real-World Effectiveness of Injection Naltrexone for Opioid Use Disorder 2. HEAL Data Platform. Study Record. 10.60490/HDP01319